# 06 Annex: will the spread widen by 25 bp?

Role 5. Due Friday 20 Nov.

Naive Bayes only classifies, so the annex asks: will this bond's spread widen by 25 bp or more by next month-end (`widens_25bp`)? Naive Bayes against the other classifiers from the course. A weak result is fine and expected (predicting moves is much harder than predicting levels); a clean comparison is the point.

Class notebooks followed: `2. Naive_Bayes_SMS` (the structure) and `1_Naive_Bayes_Tweets` (Bayes' theorem, priors, smoothing).

Every step is coded and runs top to bottom on the shared panel (about 10 minutes on a laptop). Your job: run it, read each output,
write the interpretation cells in your own words, and be able to explain every line (the professor can ask
anyone). Where a step says **Your reading**, replace the prompt with what the output shows.

Rules: validation only (`partition == "valid"`); test is opened once, in `99_final`. Shared code comes from `ml2`
and is never copied here. Figures are saved to `reports/figures/06_*.png`. Paste the lines printed at the
end into `RUN_LOG.md`. Clear the outputs before you commit (the data is licensed: no rows in Git).

## 1. Setup

Rows without a next month have no label and are dropped by `xy()`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, log_loss, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import CategoricalNB, GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, KBinsDiscretizer, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier

from ml2.config import load_config, resolve
from ml2.data import check_panel, load_panel
from ml2.features import categorical_columns, make_preprocessor, numeric_columns, xy
from ml2.splits import walk_forward_folds

cfg = load_config()
TARGET = cfg["problem"]["annex_target"]
PREFIX = "06"
FIG = resolve("reports/figures")
FIG.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    """Save a figure to reports/figures/<prefix>_<name>.png."""
    fig.savefig(FIG / f"{PREFIX}_{name}.png", dpi=150, bbox_inches="tight")


check_panel()
df = load_panel(partitions=["train", "valid"])
train, valid = df[df.partition == "train"], df[df.partition == "valid"]
X_tr, y_tr = xy(train, target=TARGET)
X_va, y_va = xy(valid, target=TARGET)
y_tr, y_va = y_tr.astype(int), y_va.astype(int)
cv = walk_forward_folds(train.loc[X_tr.index, "snapshot_date"])  # replaces StratifiedKFold in the class notebook
print(f"train {X_tr.shape}, base rate {y_tr.mean():.3f}; valid {X_va.shape}, base rate {y_va.mean():.3f}")

**Your reading.** About 1 bond in 25 widens in train, fewer in valid (a calmer period). Why is accuracy a poor metric here? (Always saying "no" is right 96% of the time.)

## 2. Reference model

Always predict the training rate. Its log loss is the bar for calibrated probabilities; its AUC is 0.5.

In [ ]:
dummy = DummyClassifier(strategy="prior").fit(X_tr, y_tr)
results = {}


def evaluate(name, p_va, threshold=None):
    """Valid metrics for predicted probabilities; precision and recall at the chosen threshold."""
    p = np.clip(p_va, 1e-6, 1 - 1e-6)
    r = {"AUC": roc_auc_score(y_va, p), "log loss": log_loss(y_va, p)}
    if threshold is not None:
        yhat = (p_va >= threshold).astype(int)
        r.update({"threshold": threshold, "precision": precision_score(y_va, yhat, zero_division=0),
                  "recall": recall_score(y_va, yhat)})
    results[name] = {**r, "p": p_va}
    return pd.Series(r).round(3)


evaluate("Reference (training rate)", dummy.predict_proba(X_va)[:, 1])

## 3. Gaussian Naive Bayes

Bayes' theorem: P(widens | features) is proportional to P(widens) x P(features | widens). "Naive": the features are
assumed independent given the class, so P(features | widens) is the product of one term per feature. Gaussian NB
models each numeric feature as a normal distribution within each class, here on scaled features.

In [ ]:
gnb = Pipeline([("pre", make_preprocessor()), ("nb", GaussianNB())]).fit(X_tr, y_tr)
evaluate("Gaussian Naive Bayes", gnb.predict_proba(X_va)[:, 1])

## 4. Categorical Naive Bayes

Our features are numbers, not words: binned into quintiles, then Categorical NB counts how often each bin occurs
in each class, the role the class's Multinomial NB on word counts played. Laplace smoothing (`alpha=1`) adds one
to every count, so a bin never seen with a 1 does not get probability zero (which would veto the whole product).

In [ ]:
def make_binned(num_cols, cat_cols):
    # impute, bin numbers into quintiles, encode categories, and shift the codes by one so an unseen category
    # becomes 0 instead of -1 (Categorical NB needs non-negative integer codes)
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("bin", KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile"))]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Unknown")),
                          ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
                          ("shift", FunctionTransformer(lambda a: a + 1))]), cat_cols),
    ])


binned = make_binned(numeric_columns(), categorical_columns())
cnb = Pipeline([("pre", binned), ("nb", CategoricalNB(alpha=1.0))]).fit(X_tr, y_tr)
evaluate("Categorical Naive Bayes", cnb.predict_proba(X_va)[:, 1])

## 5. Compare the variants

Naive_Bayes_SMS compared bag-of-words with TF-IDF by cross-validated ROC AUC; here Gaussian against Categorical NB, on the time-ordered folds.

In [ ]:
pipe = Pipeline([("pre", make_preprocessor()), ("nb", GaussianNB())])
search = GridSearchCV(pipe, [{"pre": [make_preprocessor()], "nb": [GaussianNB()]},
                             {"pre": [binned], "nb": [CategoricalNB(alpha=1.0)]}],
                      scoring="roc_auc", cv=cv, n_jobs=-1, refit=True).fit(X_tr, y_tr)
pd.DataFrame(search.cv_results_)[["param_nb", "mean_test_score", "std_test_score"]].round(3)

## 6. Choose the threshold

Naive_Bayes_SMS: the threshold is chosen on out-of-fold predictions of the training data (never on valid), at the
best F1. `cross_val_predict` needs folds that cover every row, which time-ordered folds do not, so the out-of-fold
predictions are collected with a short loop.

In [ ]:
def oof_threshold(model, X=X_tr, y=y_tr, folds=cv):
    """F1-optimal threshold from out-of-fold predictions on the training folds."""
    idx, prob = [], []
    for tr_i, va_i in folds:
        m = clone(model).fit(X.iloc[tr_i], y.iloc[tr_i])
        idx.append(va_i); prob.append(m.predict_proba(X.iloc[va_i])[:, 1])
    idx, prob = np.concatenate(idx), np.concatenate(prob)
    precision, recall, thresholds = precision_recall_curve(y.iloc[idx], prob)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    return float(thresholds[int(np.argmax(f1))]), (precision, recall, thresholds, f1)


best_threshold, (precision, recall, thresholds, f1) = oof_threshold(search.best_estimator_)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, precision[:-1], label="precision"); ax.plot(thresholds, recall[:-1], label="recall")
ax.plot(thresholds, f1, label="F1"); ax.axvline(best_threshold, color="black", ls="--", label="chosen")
ax.set_xlabel("Threshold on P(widens)"); ax.set_title("Threshold from out-of-fold predictions"); ax.legend()
save(fig, "threshold")

p_va = search.best_estimator_.predict_proba(X_va)[:, 1]
print(f"threshold {best_threshold:.3f}")
print(classification_report(y_va, (p_va >= best_threshold).astype(int), digits=3))
print(confusion_matrix(y_va, (p_va >= best_threshold).astype(int)))
# The winning variant's row in the comparison table now carries its threshold, precision and recall
nb_name = "Gaussian Naive Bayes" if isinstance(search.best_estimator_["nb"], GaussianNB) else "Categorical Naive Bayes"
evaluate(nb_name, p_va, best_threshold)

## 7. The independence problem

Highly correlated features (see 01, correlations) are counted as separate evidence, so Naive Bayes becomes
over-confident: probabilities pushed to 0 or 1, which the log loss punishes. Refit Categorical NB keeping one of
each correlated group.

In [ ]:
DROP = ["vol_20d", "vol_60d", "mom_12_1", "bbb_oas_bps", "slope_2s10s_bps"]  # keep vol_252d, ret_252d, hy_oas_bps
reduced = [c for c in numeric_columns() if c not in DROP]
cnb_reduced = Pipeline([("pre", make_binned(reduced, categorical_columns())), ("nb", CategoricalNB(alpha=1.0))])
cnb_reduced.fit(X_tr, y_tr)
print("dropped:", DROP)
evaluate("Categorical NB, one per correlated group", cnb_reduced.predict_proba(X_va)[:, 1])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name in ["Gaussian Naive Bayes", "Categorical Naive Bayes", "Categorical NB, one per correlated group"]:
    ax.hist(results[name]["p"], bins=50, histtype="step", label=name, density=True)
ax.set_yscale("log"); ax.set_xlabel("Predicted P(widens), valid"); ax.set_title("How confident is Naive Bayes?")
ax.legend(fontsize=8)
save(fig, "nb_confidence")

**Your reading.** Does removing correlated features improve the log loss? Where do Gaussian NB's probabilities pile up?

## 8. The other classifiers

Same features, folds and threshold method; small grids. KNN is fitted on a 30,000-row sample (prediction is slow
on all rows). Logistic regression is a reference, from ML I (`Logit_for_Classification`).

In [ ]:
tree_pre = lambda: make_preprocessor(scale=False, one_hot=False)  # noqa: E731
candidates = {
    "Pruned decision tree": GridSearchCV(
        Pipeline([("pre", tree_pre()), ("est", DecisionTreeClassifier(random_state=42))]),
        {"est__max_depth": [4, 6, 8], "est__min_samples_leaf": [100, 500]}, scoring="roc_auc", cv=cv, n_jobs=-1),
    "Random forest": GridSearchCV(
        Pipeline([("pre", tree_pre()), ("est", RandomForestClassifier(n_estimators=200, random_state=42,
                                                                      n_jobs=-1))]),
        {"est__min_samples_leaf": [20, 100], "est__max_features": ["sqrt", 0.3]}, scoring="roc_auc", cv=cv),
    "XGBoost": GridSearchCV(
        Pipeline([("pre", tree_pre()), ("est", xgb.XGBClassifier(n_estimators=300, subsample=0.8,
                                                                 colsample_bytree=0.7, random_state=42, n_jobs=-1))]),
        {"est__max_depth": [3, 6], "est__learning_rate": [0.05, 0.1]}, scoring="roc_auc", cv=cv),
    "Logistic regression": GridSearchCV(
        Pipeline([("pre", make_preprocessor()), ("est", LogisticRegression(max_iter=2000))]),
        {"est__C": [0.1, 1.0]}, scoring="roc_auc", cv=cv, n_jobs=-1),
}
fitted = {}
for name, gs in candidates.items():
    gs.fit(X_tr, y_tr)
    model = gs.best_estimator_
    thr, _ = oof_threshold(model)
    fitted[name] = model
    print(name, gs.best_params_, f"CV AUC {gs.best_score_:.3f}")
    display(evaluate(name, model.predict_proba(X_va)[:, 1], thr))

In [ ]:
# KNN on a 30,000-row sample of train (time order kept for its folds)
knn_rows = X_tr.sample(30000, random_state=42).index
X_k, y_k = X_tr.loc[knn_rows].sort_index(), y_tr.loc[knn_rows].sort_index()
cv_k = walk_forward_folds(train.loc[X_k.index, "snapshot_date"])
gs = GridSearchCV(Pipeline([("pre", make_preprocessor()), ("est", KNeighborsClassifier())]),
                  {"est__n_neighbors": [25, 50, 100]}, scoring="roc_auc", cv=cv_k, n_jobs=-1).fit(X_k, y_k)
thr, _ = oof_threshold(gs.best_estimator_, X_k, y_k, cv_k)
fitted["KNN"] = gs.best_estimator_
print("KNN", gs.best_params_, f"CV AUC {gs.best_score_:.3f}")
evaluate("KNN (30,000-row sample)", gs.best_estimator_.predict_proba(X_va)[:, 1], thr)

## 9. Comparison table

Valid, every model. ROC curves in one figure.

In [ ]:
table = pd.DataFrame({k: {m: v for m, v in r.items() if m != "p"} for k, r in results.items()}).T
table = table[["AUC", "log loss", "threshold", "precision", "recall"]].sort_values("AUC", ascending=False)
table.to_csv(FIG.parent / "06_comparison_valid.csv")

fig, ax = plt.subplots(figsize=(7, 6))
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_va, r["p"])
    ax.plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_va, r['p']):.2f})")
ax.plot([0, 1], [0, 1], color="grey", ls=":")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("Will the spread widen by 25 bp? ROC on valid"); ax.legend(fontsize=7)
save(fig, "roc")
table.round(3)

In [ ]:
# Which signals warn that a bond is about to widen? Permutation-free view: the random forest's importances
rf = fitted["Random forest"]
pd.Series(rf["est"].feature_importances_, index=rf["pre"].get_feature_names_out()).sort_values(
    ascending=False).head(10).round(3)

**Your reading.** Which model, if any, carries information (AUC well above 0.5, log loss below the reference)? Which signals warn that a bond is about to widen?

## 10. RUN_LOG and hand-off

One line per model. Do not evaluate on test.

In [ ]:
today = pd.Timestamp.today().strftime("%Y-%m-%d")
for name, r in table.iterrows():
    extra = "" if pd.isna(r["threshold"]) else (f", threshold {r['threshold']:.3f}: precision {r['precision']:.2f}, "
                                                f"recall {r['recall']:.2f}")
    print(f"| {today} | <your name> | 06_annex_classification | {name} | all 34 | fit on train, scored on valid | "
          f"n/a (AUC {r['AUC']:.3f}, log loss {r['log loss']:.3f}{extra}) | widens_25bp |")